# 准备数据

In [ ]:
# 加载模块
import polars as pl

from vnpy.trader.constant import Interval

from vnpy.alpha import AlphaLab

In [ ]:
import yaml
from pathlib import Path
from vnpy.trader.constant import Interval

cfg_path = Path("examples/alpha_research/workflow.yaml")
with cfg_path.open("r") as f:
    wf = yaml.safe_load(f)["workflow"]

g = wf["general"]
name: str = g["name"]
index_symbol: str = g["index_symbol"]
start: str = g["start"]
end: str = g["end"]
interval: Interval = Interval[g["interval"].upper()]
extended_days: int = g["extended_days"]
lab_dir: str = g["lab_dir"]
train_period: tuple[str, str] = tuple(g["train_period"])
valid_period: tuple[str, str] = tuple(g["valid_period"])
test_period: tuple[str, str] = tuple(g["test_period"])

steps_cfg = {s["type"]: s for s in wf["steps"]}

In [ ]:
lab: AlphaLab = AlphaLab(lab_dir)

In [ ]:
# 加载所有成分股代码
component_symbols: list[str] = lab.load_component_symbols(index_symbol, start, end)[:10]
component_symbols=[c for c in component_symbols if c not in ['PUMPUSDT.BINANCE', 'WALUSDT.BINANCE'] ]
print(component_symbols)


# 模型训练

In [ ]:
# 加载模块
import numpy as np

from vnpy.alpha import Segment, AlphaDataset, AlphaModel

from vnpy.alpha.model.models.mlp_model import MlpModel

In [ ]:
# 从文件缓存加载
dataset: AlphaDataset = lab.load_dataset(name)

In [ ]:
dataset.df.shape

In [ ]:
# 创建模型对象（从 workflow.yaml 加载配置）
m = steps_cfg.get("train_model", {"config": {}})
mc = m.get("config", {})
kwargs = {
    "input_size": mc.get("input_size", 158),
    "hidden_sizes": tuple(mc.get("hidden_sizes", [256])),
    "lr": mc.get("lr", 0.002),
    "optimizer": mc.get("optimizer", "adam"),
    "n_epochs": mc.get("n_epochs", 8000),
    "batch_size": mc.get("batch_size", 8192),
    "weight_decay": mc.get("weight_decay", 0.0002),
    "seed": mc.get("seed", 42),
    "device": mc.get("device", "cpu"),
}

model: AlphaModel = MlpModel(**kwargs)

In [ ]:
from vnpy.alpha import Segment
df_train = dataset.fetch_feat(Segment.TRAIN)
print("TRAIN shape:", df_train.shape)
print(df_train.head())

In [ ]:
# 使用数据集训练模型
# 生成切分并用切分文件训练
splits_dir = f"{lab_dir}/splits"
dataset.process_features()
model.fit(splits_dir)

In [ ]:
# 查看模型细节
model.detail()

In [ ]:
# 保存模型
lab.save_model(name, model)

# 预测信号

In [ ]:
model: AlphaModel = lab.load_model(name)

In [ ]:
# 用模型在测试集上预测（基于切分文件）
splits_dir = f"{lab_dir}/splits"
test_parquet = f"{splits_dir}/test.parquet"
pre: np.ndarray = model.predict(test_parquet)

# 加载测试集切分（包含 datetime/vt_symbol）并合并预测为信号
import pandas as pd
df_t_pd = pd.read_parquet(test_parquet)
signal = pl.DataFrame({
    "datetime": df_t_pd["datetime"],
    "vt_symbol": df_t_pd["vt_symbol"],
    "signal": pre,
})

In [ ]:
dataset.df.head()

In [ ]:
# 检查信号绩效
#dataset.show_signal_performance(signal)

In [ ]:
# 保存信号数据
lab.save_signal(name, signal)

# 策略回测

In [ ]:
# 加载模块
import importlib
from datetime import datetime

from vnpy.alpha.strategy import BacktestingEngine

import vnpy.alpha.strategy.strategies.equity_demo_strategy as equity_demo_strategy

In [ ]:
# 重载策略类
importlib.reload(equity_demo_strategy)
EquityDemoStrategy = equity_demo_strategy.EquityDemoStrategy

In [ ]:
# 从文件加载信号数据
signal = lab.load_signal(name)

In [ ]:
# 创建回测引擎对象（从 workflow.yaml 加载配置）
engine = BacktestingEngine(lab)

bt_conf = steps_cfg.get("backtesting", {"config": {}}).get("config", {})
start_dt = datetime.fromisoformat(bt_conf.get("start", test_period[0]))
end_dt = datetime.fromisoformat(bt_conf.get("end", test_period[1]))

engine.set_parameters(
    vt_symbols=component_symbols,
    interval=interval,
    start=start_dt,
    end=end_dt,
    capital=bt_conf.get("capital", 100000000),
)

setting = bt_conf.get("setting", {"top_k": 3, "n_drop": 1, "hold_thresh": 3})
engine.add_strategy(EquityDemoStrategy, setting, signal)

In [ ]:
# 执行回测任务
engine.load_data()
engine.run_backtesting()
engine.calculate_result()
engine.calculate_statistics()
engine.show_chart()

In [ ]:
# 显示超额收益分析结果
#engine.show_performance(benchmark_symbol=index_symbol)